In [13]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [14]:
load_dotenv(Path.cwd().parent / ".env")

False

In [15]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [16]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

## Save API info as JSON

Save all endpoints needed for each company

In [80]:
import json

def save_json(company_num):
    
    endpoints = {
        "profile":   get(f"/company/{company_num}"),
        "charges":   get(f"/company/{company_num}/charges"),
        "officers":  get(f"/company/{company_num}/officers"),
        "filing":    get(f"/company/{company_num}/filing-history"),
    }

    for name, data in endpoints.items():
        with open(f"company_info_json/{company_num}_{name}.json", "w") as f:
            json.dump(data, f)

## Create a dictionary of the company info from JSON

In [81]:
def test_variables(company_num):
    endpoints = ['profile', 'charges', 'officers', 'filing']
    test_dict = {}
    for name in endpoints:
        with open(f"company_info_json/{company_num}_{name}.json", "r") as f:
            test_dict[name] = json.load(f)
    return test_dict


In [83]:
company_number = '11257129'
save_json(company_number) # 200 means successfull

200
200
200
200


## Pull out variables needed

List of variables needed:

In [84]:
variables_profile = ['company_name', 'date_of_creation', 'sic_codes', 'type', 'jurisdiction', 'has_charges','has_insolvency_history']
variables_profile_acc = ['type']
variables_charges = ['status', 'created_on']


Compile them into a dictionary of the company

In [85]:
company_info = {}

# Compile company profile data
for i in variables_profile:
    company_info[i] = test_variables(company_number)['profile'][i]

# Compile locality seperately
company_info['locality'] = test_variables(company_number)['profile']['registered_office_address']['locality'] 

# Compile account type
company_info['account_type'] = test_variables(company_number)['profile']['accounts']['last_accounts']['type']

# Compile Charges (some may not have)
try:
    for i in variables_charges:
        company_info[i] = test_variables(company_number)['charges']['items'][0][i]
except IndexError:
    print(f'Skipping charges info of {test_variables(company_number)['profile']['company_name']} - list index out of range')

In [87]:
company_info # has_charges is not working (False but have outstanding charges in the charges field)

{'company_name': 'FELLTEN LTD',
 'date_of_creation': '2018-03-15',
 'sic_codes': ['29310', '29320'],
 'type': 'ltd',
 'jurisdiction': 'england-wales',
 'has_charges': False,
 'has_insolvency_history': False,
 'locality': 'Bristol',
 'account_type': 'total-exemption-full',
 'status': 'outstanding',
 'created_on': '2023-07-14'}

Next step -> dig more into charges (persons_entitled -> lender) & understand more about the charges and due_date (delivered date and stuff also to find punctuality)